
# Deep Q-Network (DQN) — Student Walkthrough Notebook

This notebook is designed to match the **3×3 GridWorld DQN video** you created in Manim.

The goal is not to build an industrial-strength DQN trainer.  
Instead, the goal is to help a student **see one DQN update at a time**:

- the current state
- the network forward pass
- the four Q-values
- the chosen action
- the TD target
- the loss
- the effect of the gradient update on `Q(s,a)`

---

## What this notebook matches from the video

- 3×3 GridWorld
- rewards:
  - normal step: `-0.04`
  - mud: `-0.5`
  - trap: `-1.0`
  - goal: `+1.0`
- neural net idea: **2 → 5 → 4**
- outputs correspond to actions: **U, D, L, R**
- a **guided four-step path** that follows the same route shown in the video:
  - `(0,0) → (1,0) → (2,0) → (2,1) → (2,2)`

---

## How to use this notebook

Run the cells **from top to bottom**.

The most important cells are:

- `begin_guided_demo()`
- `show_current_step()`
- `guided_step()`

These let you move through the video-style DQN update **one transition at a time**.

Updated note: this v2 notebook includes a safer guided-step workflow and improved figure layout so rerunning cells does not crash the notebook.


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


## 1. Tiny DQN explorer

This class contains:

- a small **online network**
- a small **target network**
- the 3×3 GridWorld
- plotting helpers for:
  - the grid
  - the network diagram
  - the Q-value bars
- step-by-step DQN update functions

The network is intentionally tiny so the logic stays visible.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle

class TinyDQNExplorer:
    def __init__(self, gamma=0.9, lr=0.10, seed=42):
        self.gamma = float(gamma)
        self.lr = float(lr)
        self.seed = int(seed)
        self.grid_size = 3
        self.rewards = np.array([
            [-0.04, -0.50, -0.04],
            [-0.04, -1.00, -0.04],
            [-0.04, -0.04,  1.00],
        ], dtype=float)
        self.goal_state = (2, 2)
        self.action_names = ["U", "D", "L", "R"]
        self.action_deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        self.guided_experiences = [
            ((0, 0), 1, -0.04, (1, 0), False),
            ((1, 0), 1, -0.04, (2, 0), False),
            ((2, 0), 3, -0.04, (2, 1), False),
            ((2, 1), 3,  1.00, (2, 2), True),
        ]
        self.reset(seed=self.seed)

    def reset(self, seed=None):
        if seed is not None:
            self.seed = int(seed)
        self.rng = np.random.default_rng(self.seed)
        self.W1 = self.rng.normal(0.0, 0.35, size=(2, 5))
        self.b1 = np.zeros(5)
        self.W2 = self.rng.normal(0.0, 0.30, size=(5, 4))
        self.b2 = np.zeros(4)
        self.sync_target_network()
        self.demo_index = 0
        self.current_state = (0, 0)
        self.path = [self.current_state]
        self.logs = []
        self.training_history = []
        self.episode_counter = 0

    def sync_target_network(self):
        self.target_W1 = self.W1.copy()
        self.target_b1 = self.b1.copy()
        self.target_W2 = self.W2.copy()
        self.target_b2 = self.b2.copy()

    def encode_state(self, state):
        r, c = state
        return np.array([r / (self.grid_size - 1), c / (self.grid_size - 1)], dtype=float)

    @staticmethod
    def relu(x):
        return np.maximum(0.0, x)

    def forward(self, state, net="online"):
        x = self.encode_state(state)
        if net == "online":
            W1, b1, W2, b2 = self.W1, self.b1, self.W2, self.b2
        elif net == "target":
            W1, b1, W2, b2 = self.target_W1, self.target_b1, self.target_W2, self.target_b2
        else:
            raise ValueError("net must be 'online' or 'target'")
        z1 = x @ W1 + b1
        h1 = self.relu(z1)
        q = h1 @ W2 + b2
        return x, z1, h1, q

    def q_values(self, state, net="online"):
        return self.forward(state, net=net)[-1].copy()

    def greedy_action(self, state, net="online"):
        q = self.q_values(state, net=net)
        return int(np.argmax(q))

    def policy_table(self):
        rows = []
        for r in range(self.grid_size):
            row = []
            for c in range(self.grid_size):
                if (r, c) == self.goal_state:
                    row.append("G")
                else:
                    row.append(self.action_names[self.greedy_action((r, c))])
            rows.append(row)
        return pd.DataFrame(rows, index=[f"r{r}" for r in range(self.grid_size)], columns=[f"c{c}" for c in range(self.grid_size)])

    def values_table(self):
        vals = np.zeros((self.grid_size, self.grid_size))
        for r in range(self.grid_size):
            for c in range(self.grid_size):
                vals[r, c] = np.max(self.q_values((r, c)))
        return pd.DataFrame(vals, index=[f"r{r}" for r in range(self.grid_size)], columns=[f"c{c}" for c in range(self.grid_size)]).round(3)

    def inspect_state(self, state):
        online = self.q_values(state, net="online")
        target = self.q_values(state, net="target")
        rows = []
        for idx, name in enumerate(self.action_names):
            ns, reward, done = self.transition_from(state, idx)
            rows.append({
                "action": name,
                "q_online": round(float(online[idx]), 4),
                "q_target_net": round(float(target[idx]), 4),
                "next_state": ns,
                "reward": reward,
                "done": done,
                "greedy_online": idx == int(np.argmax(online)),
            })
        return pd.DataFrame(rows)

    def transition_from(self, state, action_idx):
        r, c = state
        dr, dc = self.action_deltas[action_idx]
        nr, nc = r + dr, c + dc
        if 0 <= nr < self.grid_size and 0 <= nc < self.grid_size:
            ns = (nr, nc)
            reward = float(self.rewards[nr, nc])
            done = (ns == self.goal_state)
            return ns, reward, done
        return state, -0.10, False

    def compute_td_target(self, reward, next_state, done):
        if done:
            return float(reward), 0.0
        next_q_target = self.q_values(next_state, net="target")
        max_next = float(np.max(next_q_target))
        return float(reward + self.gamma * max_next), max_next

    def train_on_transition(self, state, action_idx, reward, next_state, done):
        x, z1, h1, q_before = self.forward(state, net="online")
        target, max_next = self.compute_td_target(reward, next_state, done)
        pred = float(q_before[action_idx])
        loss = float((target - pred) ** 2)

        grad_q = np.zeros(4, dtype=float)
        grad_q[action_idx] = 2.0 * (pred - target)

        W2_before = self.W2.copy()
        dW2 = np.outer(h1, grad_q)
        db2 = grad_q.copy()

        dh = W2_before @ grad_q
        dz1 = dh * (z1 > 0).astype(float)
        dW1 = np.outer(x, dz1)
        db1 = dz1.copy()

        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

        q_after = self.q_values(state, net="online")
        result = {
            "state": state,
            "action_idx": int(action_idx),
            "action": self.action_names[action_idx],
            "reward": float(reward),
            "next_state": next_state,
            "done": bool(done),
            "q_before": q_before.copy(),
            "q_after": q_after.copy(),
            "prediction_before": pred,
            "td_target": float(target),
            "max_next_target_q": float(max_next),
            "loss": float(loss),
            "delta_q_action": float(q_after[action_idx] - pred),
        }
        self.logs.append(result)
        self.current_state = next_state
        self.path.append(next_state)
        return result

    def begin_guided_demo(self):
        self.demo_index = 0
        self.current_state = (0, 0)
        self.path = [self.current_state]
        self.logs = []
        print("Guided demo reset. Current state is (0, 0). Use show_current_step() or guided_step().")

    def show_current_step(self):
        if self.demo_index >= len(self.guided_experiences):
            print("The guided video-aligned demo is finished.")
            return None
        state, action_idx, reward, next_state, done = self.guided_experiences[self.demo_index]
        q = self.q_values(state, net="online")
        target, max_next = self.compute_td_target(reward, next_state, done)
        summary = pd.DataFrame([{
            "state": state,
            "chosen_action": self.action_names[action_idx],
            "reward": reward,
            "next_state": next_state,
            "done": done,
            "predicted_q_for_action": round(float(q[action_idx]), 4),
            "max_next_q_from_target_net": round(float(max_next), 4),
            "td_target": round(float(target), 4),
        }])
        return summary

    def guided_step(self, sync_target=True, render=True):
        if self.demo_index >= len(self.guided_experiences):
            print("No guided steps left. The demo already reached the goal. Run demo.begin_guided_demo() to restart the 4-step walkthrough.")
            return None
        state, action_idx, reward, next_state, done = self.guided_experiences[self.demo_index]
        result = self.train_on_transition(state, action_idx, reward, next_state, done)
        self.demo_index += 1
        if sync_target:
            self.sync_target_network()
        if render:
            title = f"Guided step {self.demo_index}: state={state}, action={self.action_names[action_idx]}"
            self.render(state=state, chosen_action=action_idx, td_target=result["td_target"], loss=result["loss"], title=title)
        return result

    def train_episode(self, epsilon=0.20, max_steps=12, sync_every_step=False):
        state = (0, 0)
        path = [state]
        total_reward = 0.0
        step_count = 0
        while step_count < max_steps:
            if self.rng.random() < epsilon:
                action = int(self.rng.integers(4))
            else:
                action = self.greedy_action(state)
            next_state, reward, done = self.transition_from(state, action)
            self.train_on_transition(state, action, reward, next_state, done)
            if sync_every_step:
                self.sync_target_network()
            state = next_state
            path.append(state)
            total_reward += reward
            step_count += 1
            if done:
                break
        self.episode_counter += 1
        hist = {"episode": self.episode_counter, "steps": step_count, "return": total_reward, "epsilon": epsilon, "path": path}
        self.training_history.append(hist)
        return hist

    def train(self, episodes=40, epsilon_start=0.30, epsilon_decay=0.95, epsilon_min=0.05, sync_every=4):
        epsilon = float(epsilon_start)
        for ep in range(1, episodes + 1):
            sync_each = False
            out = self.train_episode(epsilon=epsilon, sync_every_step=sync_each)
            if ep % sync_every == 0:
                self.sync_target_network()
            epsilon = max(epsilon_min, epsilon * epsilon_decay)
        return pd.DataFrame(self.training_history)

    def greedy_path(self, max_steps=12):
        state = (0, 0)
        path = [state]
        visited = {state}
        reason = "max_steps"
        for _ in range(max_steps):
            if state == self.goal_state:
                reason = "goal"
                break
            action = self.greedy_action(state)
            next_state, reward, done = self.transition_from(state, action)
            path.append(next_state)
            if done:
                reason = "goal"
                break
            if next_state in visited:
                reason = "loop"
                break
            visited.add(next_state)
            state = next_state
        return path, reason

    def result_table(self, result):
        if result is None:
            return pd.DataFrame({
                'message': [
                    "No new guided result is available. Run demo.begin_guided_demo() to restart the 4-step walkthrough."
                ]
            })
        return pd.DataFrame({
            'action': self.action_names,
            'q_before': np.round(result['q_before'], 4),
            'q_after': np.round(result['q_after'], 4),
            'delta': np.round(result['q_after'] - result['q_before'], 4),
        })

    def result_summary(self, result):
        if result is None:
            return pd.DataFrame({
                'message': [
                    "No new guided result is available. Run demo.begin_guided_demo() to restart the 4-step walkthrough."
                ]
            })
        return pd.DataFrame([{
            "state": result["state"],
            "action": result["action"],
            "reward": result["reward"],
            "next_state": result["next_state"],
            "done": result["done"],
            "td_target": round(result["td_target"], 6),
            "loss": round(result["loss"], 6),
        }])

    def recent_log(self, n=5):
        if not self.logs:
            return pd.DataFrame()
        rows = []
        for item in self.logs[-n:]:
            rows.append({
                "state": item["state"],
                "action": item["action"],
                "reward": round(item["reward"], 3),
                "next_state": item["next_state"],
                "td_target": round(item["td_target"], 4),
                "prediction_before": round(item["prediction_before"], 4),
                "loss": round(item["loss"], 6),
                "delta_q_action": round(item["delta_q_action"], 4),
            })
        return pd.DataFrame(rows)

    def _cell_color(self, reward):
        if reward == 1.0:
            return "#0f8f73"
        if reward == -1.0:
            return "#8b1e1e"
        if reward == -0.5:
            return "#e96d0c"
        return "#3d4a5d"

    def _draw_grid(self, ax, state=None, path=None, show_policy=True):
        n = self.grid_size
        ax.set_xlim(0, n)
        ax.set_ylim(0, n)
        ax.set_aspect("equal")
        ax.invert_yaxis()
        ax.set_xticks(np.arange(n) + 0.5, [f"c{c}" for c in range(n)])
        ax.set_yticks(np.arange(n) + 0.5, [f"r{r}" for r in range(n)])
        for r in range(n):
            for c in range(n):
                rect = Rectangle((c, r), 1, 1, facecolor=self._cell_color(float(self.rewards[r, c])), edgecolor="white", linewidth=2)
                ax.add_patch(rect)
                rew = self.rewards[r, c]
                top = ""
                if (r, c) == self.goal_state:
                    top = "Goal\n+1.0"
                elif rew == -1.0:
                    top = "Trap\n-1.0"
                elif rew == -0.5:
                    top = "Mud\n-0.5"
                else:
                    top = "-0.04"
                ax.text(c + 0.5, r + 0.18, top, ha="center", va="center", fontsize=10, color="white")
                max_q = np.max(self.q_values((r, c)))
                ax.text(c + 0.5, r + 0.63, f"{max_q:.3f}", ha="center", va="center", fontsize=14, color="white")
                if show_policy and (r, c) != self.goal_state:
                    a = self.greedy_action((r, c))
                    dxdy = {0:(0,-0.23),1:(0,0.23),2:(-0.23,0),3:(0.23,0)}[a]
                    ax.arrow(c + 0.5, r + 0.5, dxdy[0], dxdy[1], width=0.02, head_width=0.12, head_length=0.10, length_includes_head=True, color="gold")
        if path and len(path) >= 2:
            xs = [c + 0.5 for _, c in path]
            ys = [r + 0.5 for r, _ in path]
            ax.plot(xs, ys, marker="o", linewidth=2.5)
        if state is not None:
            sr, sc = state
            ax.add_patch(Rectangle((sc, sr), 1, 1, fill=False, edgecolor="yellow", linewidth=3))
            ax.scatter([sc + 0.5], [sr + 0.5], s=180, zorder=5)
        ax.set_title("GridWorld")
        ax.set_facecolor("#222222")

    def _draw_network(self, ax, state, q_values):
        ax.set_title("Tiny DQN (2 → 5 → 4)", fontsize=15, pad=10)
        ax.axis("off")
        ax.set_xlim(0.0, 1.0)
        ax.set_ylim(0.0, 1.0)
        layer_sizes = [2, 5, 4]
        x_positions = [0.15, 0.5, 0.85]
        layer_ys = {
            2: np.linspace(0.68, 0.34, 2),
            5: np.linspace(0.86, 0.14, 5),
            4: np.linspace(0.76, 0.24, 4),
        }
        node_pos = []
        for li, size in enumerate(layer_sizes):
            xs = np.full(size, x_positions[li])
            ys = layer_ys[size]
            node_pos.append(list(zip(xs, ys)))
        for p1 in node_pos[0]:
            for p2 in node_pos[1]:
                ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="0.78", linewidth=0.8, alpha=0.7)
        for p1 in node_pos[1]:
            for p2 in node_pos[2]:
                ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="0.78", linewidth=0.8, alpha=0.7)
        x_feat = self.encode_state(state)
        h = self.forward(state)[2]
        outputs = np.array(q_values)
        for idx, (x, y) in enumerate(node_pos[0]):
            color = "#ffd54f"
            circ = Circle((x, y), 0.035, facecolor=color, edgecolor="black", linewidth=1.2)
            ax.add_patch(circ)
            ax.text(x - 0.08, y, ["r", "c"][idx], va="center", ha="right", fontsize=11)
            ax.text(x, y - 0.075, f"{x_feat[idx]:.2f}", va="top", ha="center", fontsize=10)
        for idx, (x, y) in enumerate(node_pos[1]):
            intensity = min(1.0, 0.25 + float(h[idx]) * 2.5)
            circ = Circle((x, y), 0.03, facecolor=(0.55, 0.80, 1.0, intensity), edgecolor="black", linewidth=1.0)
            ax.add_patch(circ)
        for idx, (x, y) in enumerate(node_pos[2]):
            chosen = idx == int(np.argmax(outputs))
            face = "#ffe082" if chosen else "#a5d6a7"
            circ = Circle((x, y), 0.035, facecolor=face, edgecolor="black", linewidth=1.2)
            ax.add_patch(circ)
            ax.text(x + 0.08, y, self.action_names[idx], va="center", ha="left", fontsize=11)
            ax.text(x, y - 0.075, f"{outputs[idx]:.2f}", va="top", ha="center", fontsize=10)
        ax.text(
            0.5,
            0.94,
            "Forward pass: state features → hidden activations → Q-values",
            ha="center",
            va="top",
            fontsize=10,
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", alpha=0.75),
        )

    def _draw_q_bars(self, ax, q_values, chosen_action=None, td_target=None, loss=None):
        q_values = np.array(q_values, dtype=float)
        xs = np.arange(4)
        colors = ["#7bc67b"] * 4
        if chosen_action is not None:
            colors[chosen_action] = "#ffd54f"
        ax.bar(xs, q_values, color=colors, width=0.6)
        ymax = max(1.25, float(np.max(q_values)) + 0.4, float(td_target) + 0.3 if td_target is not None else 1.0)
        ymin = min(-0.25, float(np.min(q_values)) - 0.2)
        ax.set_ylim(ymin, ymax)
        ax.set_xticks(xs, self.action_names)
        ax.axhline(0.0, color="black", linewidth=1)
        if td_target is not None and chosen_action is not None:
            ax.axhline(td_target, color="red", linestyle="--", linewidth=2, label=f"TD target = {td_target:.3f}")
            pred = q_values[chosen_action]
            ax.plot([chosen_action, chosen_action], [pred, td_target], color="red", linewidth=3)
            if loss is not None:
                label_y = max(pred, td_target) + 0.08
                label_y = min(label_y, ymax - 0.06)
                ax.text(
                    chosen_action + 0.14,
                    label_y,
                    f"loss={loss:.3f}",
                    color="red",
                    fontsize=10,
                    bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.75),
                )
            ax.legend(loc="upper left", fontsize=9)
        for x, val in zip(xs, q_values):
            ax.text(x, val + (0.03 if val >= 0 else -0.06), f"{val:.2f}", ha="center", va="bottom" if val >= 0 else "top", fontsize=10)
        ax.set_title("Q-values for the current state")
        ax.set_ylabel("Q(s, a)")

    def render(self, state=None, chosen_action=None, td_target=None, loss=None, title="DQN walkthrough", path=None):
        if state is None:
            state = self.current_state
        q = self.q_values(state)
        fig, axes = plt.subplots(
            1,
            3,
            figsize=(16.2, 4.9),
            gridspec_kw={"width_ratios": [1.15, 1.08, 1.12]},
            constrained_layout=True,
        )
        self._draw_grid(axes[0], state=state, path=path if path is not None else self.path, show_policy=True)
        self._draw_network(axes[1], state=state, q_values=q)
        self._draw_q_bars(axes[2], q_values=q, chosen_action=chosen_action, td_target=td_target, loss=loss)
        fig.suptitle(title, fontsize=17, y=1.03)
        plt.show()

    def plot_training_history(self):
        if not self.training_history:
            print("No training history yet. Run train_episode() or train() first.")
            return
        df = pd.DataFrame(self.training_history)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(df["episode"], df["steps"], marker="o")
        axes[0].set_title("Steps per episode")
        axes[0].set_xlabel("Episode")
        axes[0].set_ylabel("Steps")
        axes[1].plot(df["episode"], df["return"], marker="o")
        axes[1].set_title("Return per episode")
        axes[1].set_xlabel("Episode")
        axes[1].set_ylabel("Return")
        fig.tight_layout()
        plt.show()

    def plot_video_demo_path(self):
        video_path = [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2)]
        fig, ax = plt.subplots(figsize=(4.5, 4.5))
        self._draw_grid(ax, state=video_path[-1], path=video_path, show_policy=True)
        ax.set_title("Video-aligned final path")
        plt.show()
        return video_path

    def plot_greedy_run(self, max_steps=10, title=None):
        path, reason = self.greedy_path(max_steps=max_steps)
        state = path[min(len(path)-1, 0)]
        if title is None:
            title = f"Greedy run on current network (stop reason: {reason})"
        fig, ax = plt.subplots(figsize=(4.5, 4.5))
        self._draw_grid(ax, state=path[-1], path=path, show_policy=True)
        ax.set_title(title)
        plt.show()
        return path, reason






## 2. Create the demo object

This uses the same core setting as the video:

- `gamma = 0.9`
- `seed = 42`

You can also adjust the learning rate if you want to experiment.


In [ ]:

demo = TinyDQNExplorer(gamma=0.9, lr=0.10, seed=42)

demo.render(title="Initial network before any DQN update")
display(demo.values_table())
display(demo.policy_table())



## 3. Inspect the starting state

Before updating anything, look at the current Q-values for a single state.

Try the start state `(0, 0)` first.


In [ ]:

display(demo.inspect_state((0, 0)))



## 4. Reset and start the guided video-aligned demo

This puts the walkthrough back at the first transition of the video path.


In [ ]:

demo.begin_guided_demo()


### Rerun note

The guided walkthrough has **exactly 4 steps**. If you rerun a step cell after the demo already reached the goal, the notebook will now show a friendly message instead of crashing. To replay the sequence from the start, run `demo.begin_guided_demo()` again.



## 5. Preview the next DQN update before applying it

This cell shows the **current transition**:

- current state
- chosen action
- reward
- next state
- predicted Q-value for the chosen action
- target-network bootstrap value
- TD target

At this moment, the network has **not** been updated yet.


In [ ]:
preview = demo.show_current_step()
display(preview)
if preview is not None:
    row = preview.iloc[0]
    action_idx = demo.action_names.index(row["chosen_action"])
    demo.render(
        state=row["state"],
        chosen_action=action_idx,
        td_target=row["td_target"],
        title="Preview before guided step 1",
    )
else:
    print("The guided demo is already finished. Run demo.begin_guided_demo() to restart it.")



## 6. Run guided step 1

This performs **one DQN update** on the first transition.

Look carefully at:

- `q_before`
- `q_after`
- `delta`
- `td_target`
- `loss`


In [ ]:
 = demo.guided_step(sync_target=True, render=True)
display(demo.result_table())
display(demo.result_summary())



## 7. Run guided step 2


In [ ]:
 = demo.guided_step(sync_target=True, render=True)
display(demo.result_table())
display(demo.result_summary())



## 8. Run guided step 3


In [ ]:
 = demo.guided_step(sync_target=True, render=True)
display(demo.result_table())
display(demo.result_summary())



## 9. Run guided step 4 (goal transition)

This is the last transition in the guided video path.

Because the next state is the goal, the TD target becomes just the terminal reward.


In [ ]:
 = demo.guided_step(sync_target=True, render=True)
display(demo.result_table())
display(demo.result_summary())



## 10. Review what the network learned from the four guided updates

The tables below summarize:

- the current greedy action in every cell
- the current max Q-value in every cell
- the recent DQN update log


In [ ]:

display(demo.policy_table())
display(demo.values_table())
display(demo.recent_log())



## 11. Compare different states after the guided demo

This is useful for asking students questions like:

- Why is the value near the goal larger?
- Why is the mud state unattractive?
- Why do some cells still have weak or noisy policies?

Try different states such as `(2, 1)`, `(1, 1)`, `(0, 2)`.


In [ ]:

display(demo.inspect_state((2, 1)))
display(demo.inspect_state((1, 1)))
display(demo.inspect_state((0, 2)))



## 12. Video-aligned final path

The original video ends with a clean path:

`(0,0) → (1,0) → (2,0) → (2,1) → (2,2)`

The plot below shows that path directly on the grid.

This is helpful for students who want to connect the notebook back to the animation.


In [ ]:

demo.plot_video_demo_path()



## 13. Optional experiment: train the tiny DQN for several episodes

This section is **optional**.

It lets students experiment with repeated DQN-style updates on the same environment.
Because this is a very small teaching network, the result is not guaranteed to be perfectly optimal every time.

What matters here is understanding:

- repeated Bellman-style targets
- online network vs. target network
- how the policy can change over time


In [ ]:

demo.reset(seed=42)

history = demo.train(
    episodes=30,
    epsilon_start=0.30,
    epsilon_decay=0.95,
    epsilon_min=0.05,
    sync_every=4
)

display(history.tail())
demo.plot_training_history()
display(demo.policy_table())
display(demo.values_table())



## 14. Optional experiment: greedy run on the current network

This shows the path produced by the **current learned policy** of the tiny network.

Sometimes it reaches the goal.  
Sometimes it loops or stops because the tiny network is only an educational approximation.

That is okay for this notebook: the main learning target is the **mechanics of a DQN update**, not benchmark performance.


In [ ]:

path, reason = demo.plot_greedy_run(max_steps=10)
print("Greedy path:", path)
print("Stop reason:", reason)



## 15. Suggested student questions

1. In step 1, why does only one action value change more than the others?
2. Why is the goal transition different from the non-terminal transitions?
3. What role does the **target network** play in the TD target?
4. Why can a tiny neural network produce imperfect policies even in a very small grid?
5. How is this notebook different from tabular Q-learning?

---

## 16. Extension ideas

You can extend this notebook by adding:

- experience replay
- a separate replay buffer viewer
- one-hot state encoding
- a larger hidden layer
- a real PyTorch implementation
- a comparison between **Q-learning** and **DQN**
